# Open Hit Expander + Boltz-2 — v0.7 compound assessment

Full run or timestamp reload, representative pose viewer, PLIP interaction analysis, stability plots, and a final compound assessment report.


## 1. Install and select workflow

In [ ]:
# Run from the repository root in the Jupyter environment:
# %pip install -e .

from pathlib import Path
import os
import shutil
import subprocess
import pandas as pd
from IPython.display import display

from open_hit_expander import (
    HitExpanderConfig, SCREENING_PRESETS, prepare_boltz2_batch,
    run_boltz2_replicates, collect_boltz2_replicate_results,
    load_boltz2_run,
)
from open_hit_expander.widgets import hit_expander_widget
from open_hit_expander.visualization import complex_browser
from open_hit_expander.interactions import analyze_interactions, display_interaction_heatmap
from open_hit_expander.plotting import plot_interaction_summary
from open_hit_expander.stability import plot_stability

# Set to "new" to run Boltz-2; "reload" to analyze a completed run.
RUN_MODE = "reload"  # "new" or "reload"
SAVED_TIMESTAMP = ""  # e.g. 20261013_111348_145796
OUTPUT_DIRECTORY = Path("hit_expander_output")
N_REPLICATES = 2  # e.g. 1 for quick, 5 for confirmation

assert RUN_MODE in {"new", "reload"}
assert N_REPLICATES >= 1
print("Mode:", RUN_MODE)


## 2. Hit expansion — new run only

In [ ]:
INPUT = "CN(C)C[C@@H](O)COc1ccc(Nc2nccc(Nc3cc(Cl)ccc3Cl)n2)cc1"
CONFIG = HitExpanderConfig()
RUN_STATE = {"run": None, "output_files": None}

if RUN_MODE == "new":
    display(hit_expander_widget(INPUT, CONFIG, OUTPUT_DIRECTORY, state=RUN_STATE))
    print("Click Enumerate analogues before running Section 6.4.")
else:
    print("Saved run selected: hit enumeration skipped.")


## 6.2 Boltz-2 settings — new run only

In [ ]:
# Public test target: canonical chain-A sequence from human CDK2 PDB 1H01.
# The deposited construct contains a modified lysine (KCX); Boltz receives the
# canonical amino-acid sequence with Lys at that position.
PROTEIN_SEQUENCE = (
    "MENFQKVEKIGEGTYGVVYKARNKLTGEVVALKKIRLDTETEGVPSTAIREISLLKELNHPNIVKLLDVIHTENKLYLVF"
    "EFLHQDLKKFMDASALTGIPLPLIKSYLFQLLQGLAFCHSHRVLHRDLKPQNLLINTEGAIKLADFGLARAFGVPVRTYT"
    "HEVVTLWYRAPEILLGCKYYSTAVDIWSLGCIFAEMVTRRALFPGDSEIDQLFRIFRTLGTPDEVVWPGVTSMPDYKPSF"
    "PKWARQDFSKVVPPLDEDGRSLLSQMLHYDPNKRISAKAALAHPFFQDVTKPVPHLRL"
)

# Local executable path for development; restore automatic detection later.
# Resolve from override, PATH, or a separate conda environment.
BOLTZ_EXECUTABLE = os.environ.get("BOLTZ_EXECUTABLE") or shutil.which("boltz")
if RUN_MODE == "new" and not BOLTZ_EXECUTABLE:
    conda = shutil.which("conda")
    if conda:
        found = subprocess.run([conda, "run", "-n", "boltz2", "which", "boltz"], capture_output=True, text=True)
        if found.returncode == 0:
            BOLTZ_EXECUTABLE = found.stdout.strip()
if RUN_MODE == "new" and not BOLTZ_EXECUTABLE:
    raise FileNotFoundError("Boltz executable not found: activate environment or set BOLTZ_EXECUTABLE")

# Reuse one target MSA for every analogue, or leave None to use the MSA server.
# Reusing a local A3M avoids repeated server calls on later batches.
MSA_PATH = None
# MSA_PATH = Path("/absolute/path/to/cdk2_1h01.a3m")

# A laptop screen should be bounded. Set None only when the selected set is small.
MAX_ANALOGUES = 25

# Apple Silicon MPS settings.
ACCELERATOR = "gpu"
NUM_DEVICES = 1
NO_KERNELS = True
NUM_WORKERS = 0
PREPROCESSING_THREADS = 8

# Select quick, balanced, or confirm. Balanced is suitable for routine ranking.
SCREENING_PRESET = "balanced"
# SCREENING_PRESETS is imported from the package.

# Keep False for comparison with the uncorrected affinity head.
USE_AFFINITY_MW_CORRECTION = False

# Populated by the preparation, parsing, and ranking cells.
BOLTZ2_STATE = {
    "root": None,
    "job_directory": None,
    "output_directory": None,
    "manifest": None,
    "command": None,
    "screening_preset": None,
    "screening_settings": None,
    "results": None,
    "ranked": None,
}

## 6.4 Prepare the Boltz-2 batch — new run only

In [ ]:
if RUN_MODE == "new":
    if RUN_STATE.get("run") is None:
        raise RuntimeError("Enumerate your analogues in Section 2 before preparing Boltz jobs")
    prepared_batch = prepare_boltz2_batch(
        output_directory=OUTPUT_DIRECTORY,
        hit_run=RUN_STATE.get("run"),
        protein_sequence=PROTEIN_SEQUENCE,
        boltz_executable=BOLTZ_EXECUTABLE,
        msa_path=MSA_PATH,
        max_analogues=MAX_ANALOGUES,
        screening_preset=SCREENING_PRESET,
        screening_presets=SCREENING_PRESETS,
        diffusion_samples=SCREENING_PRESETS[SCREENING_PRESET]["diffusion_samples"],
        accelerator=ACCELERATOR,
        devices=NUM_DEVICES,
        no_kernels=NO_KERNELS,
        num_workers=NUM_WORKERS,
        preprocessing_threads=PREPROCESSING_THREADS,
        affinity_mw_correction=USE_AFFINITY_MW_CORRECTION,
    )
    BOLTZ2_STATE.update(prepared_batch)

    display(BOLTZ2_STATE["manifest"])
    print("Jobs:", len(BOLTZ2_STATE["manifest"]))
    print("Screening preset:", BOLTZ2_STATE["screening_preset"])
    print("Settings:", BOLTZ2_STATE["screening_settings"])
    print("Run directory:", BOLTZ2_STATE["root"].resolve())
    print("Command:")
    print(" ".join(BOLTZ2_STATE["command"]))
else:
    print("Reload mode: no new job directory created.")


## 6.5 Run all cofolding and potency replicates — new run only

This is the **only inference step**. Different seeds each generate a complex and one aggregate affinity prediction. Saved results are used downstream. Running this cell in reload mode never launches Boltz.

In [ ]:
if RUN_MODE == "new":
    replicate_predictions, replicate_status = run_boltz2_replicates(
        BOLTZ2_STATE, n_replicates=N_REPLICATES
    )
    display(replicate_status)
else:
    print("Reload mode: no inference performed.")


## 6.7 Aggregate / reload saved predictions

In `RUN_MODE = "reload"`, this reads existing inference results only. It does not perform cofolding.


In [ ]:
if RUN_MODE == "reload":
    BOLTZ2_STATE, results, ranked = load_boltz2_run(
        SAVED_TIMESTAMP, output_directory=OUTPUT_DIRECTORY,
        reparse=True,  # repairs legacy missing_prediction_directory errors
    )
else:
    results, ranked = collect_boltz2_replicate_results(BOLTZ2_STATE)

RUN_ROOT = Path(BOLTZ2_STATE["root"])
cols = ["job_id", "is_parent", "predicted_pic50", "mean_pic50",
        "sd_pic50", "min_pic50", "max_pic50", "median_delta_pic50",
        "sd_delta_pic50", "fraction_predicted_improvement", "n_success",
        "representative_replicate", "representative_seed", "representative_pic50",
        "structure_path", "cofold_status"]
display(results[[c for c in cols if c in results.columns]])
print("Loaded run:", RUN_ROOT)
print("Successful poses:", results["structure_path"].notna().sum())


## 7. Visualize median and highest-pIC50 replicate poses

The median pose is the **existing structure** whose replicate pIC50 is closest to the median. The overlay uses the highest-pIC50 replicate. With only two replicates these may coincide.


In [ ]:
# Add comparison metadata BEFORE instantiating the viewer.
# Uses existing files only. No Boltz inference.
import json
from pathlib import Path
import pandas as pd

run_root = Path(BOLTZ2_STATE["root"])
replicate_dirs = sorted((run_root / "replicates").glob("replicate_*_seed_*"))
if not replicate_dirs:
    raise FileNotFoundError(f"No replicate directories under {run_root}")

comparison_rows = []
for job_id in results["job_id"].astype(str):
    candidates = []
    for rep_dir in replicate_dirs:
        prediction_dir = rep_dir / "results" / "boltz_results_jobs" / "predictions" / job_id
        affinity_file = prediction_dir / f"affinity_{job_id}.json"
        structures = sorted(prediction_dir.glob(f"{job_id}_model_*.cif"))
        if not affinity_file.exists() or not structures:
            continue
        value = json.loads(affinity_file.read_text()).get("affinity_pred_value")
        if value is None:
            continue
        parts = rep_dir.name.split("_")
        candidates.append({
            "pic50": 6.0 - float(value),
            "structure_path": str(structures[0].resolve()),
            "replicate": int(parts[1]),
            "seed": int(parts[-1]),
        })
    if not candidates:
        continue
    best = max(candidates, key=lambda x: (x["pic50"], -x["replicate"]))
    comparison_rows.append({
        "job_id": job_id,
        "best_pic50": best["pic50"],
        "best_structure_path": best["structure_path"],
        "best_replicate": best["replicate"],
        "best_seed": best["seed"],
    })

best_df = pd.DataFrame(comparison_rows, columns=[
    "job_id", "best_pic50", "best_structure_path", "best_replicate", "best_seed"
])
results = results.drop(columns=[
    "best_pic50", "best_structure_path", "best_replicate", "best_seed"
], errors="ignore").merge(best_df, on="job_id", how="left")

print("Best-pose paths found:", results["best_structure_path"].notna().sum(), "/", len(results))
display(results[["job_id", "predicted_pic50", "representative_pic50", "best_pic50", "best_structure_path"]])

# Render ONCE after the data has been attached.
display(complex_browser(results))


## 8. PLIP analysis on selected representative poses

Only the representative structure for each ligand is analyzed. Results are saved **inside the selected run**, preventing different runs from overwriting each other. This step does not run Boltz.

In [ ]:
ANALYSIS_DIR = RUN_ROOT / "interaction_analysis"
RUN_PLIP = (RUN_MODE == "new")  # Saved PLIP CSVs are reused in reload mode
if RUN_PLIP:
    if not shutil.which("plip"):
        raise RuntimeError("PLIP is not available in the current Jupyter kernel PATH")
    interaction_details, interaction_fingerprints, interaction_status, heatmap_path = analyze_interactions(
        results, output_dir=ANALYSIS_DIR,
        protein_chain="A", ligand_chain="B", plip_executable="plip"
    )
    display(interaction_status)
    display_interaction_heatmap(interaction_details, interaction_status)
else:
    print("PLIP skipped. Set RUN_PLIP = True to analyze representative structures.")


## 9. Interaction fingerprint plots — notebook output

In [ ]:
if not (ANALYSIS_DIR / "interaction_details.csv").exists():
    print("No interaction analysis found; run Section 8 first.")
else:
    interaction_figures = plot_interaction_summary(
        details=ANALYSIS_DIR, results=results,
        ligand_sort="predicted_pic50", residue_sort="sequence"
    )


## 10. Prediction stability summary and plots — NO new inference

Show potency distributions, paired analogue-versus-parent ΔpIC50, and between-seed SD. With one replicate, SD cannot be estimated.

In [ ]:
if (RUN_ROOT / "replicate_predictions.csv").exists():
    stable_cols = ["job_id", "median_pic50", "mean_pic50", "sd_pic50",
                   "median_delta_pic50", "sd_delta_pic50", "fraction_predicted_improvement",
                   "n_success", "n_paired"]
    display(results[[c for c in stable_cols if c in results.columns]])
    n_seeds = pd.read_csv(RUN_ROOT / "replicate_predictions.csv")["seed"].nunique()
    if n_seeds > 1:
        stability_figures = plot_stability(RUN_ROOT)
    else:
        print("Only one replicate: cannot estimate between-seed SD.")
else:
    print("No replicate CSV found.")


## 10.1. Files written

The same `RUN_ROOT` is used for new and reloaded experiments. Saved outputs include per-seed Boltz JSON/mmCIF files, replicate CSVs, the aggregated compound table, the representative pose paths, and the PLIP/interaction plots data. There is no second Boltz call after Section 6.5.

## 11. Compound assessment report

Generate the final sortable compound ranking, potency-stability plots and per-ligand interaction changes using the **existing** `results` table and saved PLIP outputs. No Boltz-2 inference is performed.


In [ ]:
from open_hit_expander.report import build_compound_assessment
from IPython.display import display, HTML

REPORT_DIR = RUN_ROOT / "compound_assessment"
compound_summary, compound_report = build_compound_assessment(
    results,
    details=ANALYSIS_DIR / "interaction_details.csv",
    status=ANALYSIS_DIR / "interaction_status.csv",
    replicate_predictions=RUN_ROOT / "replicate_predictions.csv",
    output_dir=REPORT_DIR,
)
display(compound_summary)
print("HTML report:", compound_report.resolve())
display(HTML(f'<a href="{compound_report.resolve().as_uri()}" target="_blank">Open compound assessment</a>'))
